## Name:_____________________________

## **Homework Assignment**
### End-to-End EEG Preprocessing Pipeline


**Objective:** Independently execute a complete preprocessing pipeline on raw EEG data, from filtering to artifact reconstruction, and extract a clean Event-Related Potential (ERP).

**Dataset:** MNE Sample Dataset (`sample_audvis_raw.fif`)

Before tackling the exercises below, you must independently execute a complete preprocessing pipeline on the raw data.

**Pipeline Requirements:**
1. **Continuous Filtering:** Apply a 60 Hz notch filter and a 1.0 to 40.0 Hz bandpass filter to the continuous `raw` data to ensure mathematical stationarity.
2. **Epoching:** Extract the event triggers (`STI 014`). Epoch the data specifically for the `Auditory/Right` condition (Event code `2`). Set the time window from -200 ms to 500 ms. Apply a pre-stimulus baseline correction and a peak-to-peak rejection threshold of 150 µV.
3. **Artifact Rejection (ICA):** Initialize a FastICA model with 15 components. Fit the model on your filtered continuous data.
4. **Automated Detection:** Run MNE's automated EOG detection function to identify the ocular artifact component.
5. **Reconstruction:** Exclude the identified artifact, apply the ICA solution to a copy of your epochs, and re-apply the baseline correction.
6. **Visualization:** Compute and plot the average Evoked response for the `Auditory/Right` condition before and after ICA cleaning.

Once your pipeline is successfully running, use your variables to complete Exercises 3 through 8.


**Task Requirements:**
1. **The Mathematical Cost of Unmixing (Written Response):**
    * Extract the numerical data matrices from your clean and dirty Evoked objects using `.get_data()`. Compute the global variance of both using `np.var()`.
    * *Question:* State the two variance values. Mathematically, why must $Var(X_{clean})$ always be less than $Var(X_{dirty})$ if a component was successfully removed?
2. **Frequency Verification (Written Response):**
    * Compute and plot the Power Spectral Density (PSD) for a frontal electrode (`EEG 001`) and an occipital electrode (`EEG 059`) for both your clean and dirty data.
    * *Question:* Describe the difference in low-frequency power (1-5 Hz) between the dirty and clean data at the frontal versus occipital electrodes. Does this confirm that our spatial filter preserved the brain data?

In [ ]:
!pip install mne

In [ ]:
# Setup
import mne
import numpy as np
import matplotlib.pyplot as plt
from mne.preprocessing import ICA

mne.set_log_level('WARNING')

In [ ]:
#@title Data Loading
data_path = mne.datasets.sample.data_path()
raw_file_name = data_path /'MEG' / 'sample' / 'sample_audvis_raw.fif'

raw = mne.io.read_raw_fif(raw_file_name, preload=True)
raw.pick_types(meg=False, eeg=True, eog=True, stim=True)

<Raw | sample_audvis_raw.fif, 69 x 166800 (277.7 s), ~90.8 MiB, data loaded>

In [ ]:
#@title Exercise 1: Epoch Rejection Trade-off
"""
Objective: Observe what happens when threshold for epoch rejction is
too strict or too lenient

Event-Related Potentials (ERPs) require a high Signal-to-Noise Ratio (SNR). SNR
improves by averaging *more* trials, but degrades if you include *noisy* trials.

Tasks:
1. Create `epochs_strict` by setting the rejection threshold to 40 µV.
2. Create `epochs_loose` by setting the rejection threshold to 500 µV.
3. Print the number of retained epochs for both.
4. Discussion: Which threshold yields a more reliable ERP? Why is a threshold
of 40 µV potentially harmful to your analysis?
"""

In [ ]:
# Define strict threshold (eg 40 microvolts) and loose threshold (eg 500microvolts)
# Remember: MNE expects values in Volts, so utilise exponents when expressing microvoltages

strict_criteria = dict(eeg=...)
loose_criteria = dict(eeg=...)

In [ ]:
# Extract epochs using strict criteria
epochs_strict = mne.Epochs(raw, events, event_id=events_dict, tmin=tmin, tmax=tmax,
                           baseline=baseline, preload=True,
                           reject=...)

In [ ]:
# Extract epochs using loose criteria
epochs_loose = mne.Epochs(raw, events, event_id=events_dict, tmin=tmin, tmax=tmax,
                           baseline=baseline, preload=True,
                           reject=...)

In [ ]:
# Compare trial counts
print(f"Strict epochs retained: {len(...)}")
print(f"Loose epochs retained: {len(...)}")

In [ ]:
#@title Exercise 2: Anatomy of an Artifact
"""
Objective: How does an expert (or an algorithm) know an Independent Component
is an eye blink just by looking at it? It is not just about the spatial
location on the scalp; it is also about the *frequency*.

Tasks:
1. Use the `ica.plot_properties()` method to inspect the EOG component we
identified in Session 7.
2. Discussion: Look at the Power Spectral Density (PSD) plot generated by
this command. What frequency band (Delta, Theta, Alpha, Beta, or Gamma)
contains the most power for this artifact? How does this explain why we used
ICA instead of a simple low-pass filter?
"""

In [ ]:
# Plot properties of identified EOG component
# Recall: We stored the index of the blink component in the 'eog_indices' variable

fig = ica.plot_properties(raw, picks=...)

In [ ]:
#@title Exercise 3: The Mathematical Cost of Unmixing
"""
ICA does not "clean" data by adding missing information; it strictly subtracts variance. The observed scalp data matrix is $X = AS$. When we eliminate an artifact component, we set its corresponding row in $S$ to zero before multiplying back by the mixing matrix $A$.

Think of this like a blocked spike in volleyball. The total energy (variance) of the play is a combination of the spiker's power and the blocker's deflection. If your setter (the ICA algorithm) perfectly isolates the blocker and removes them from the equation, the total energy of the ball crossing the net (the reconstructed signal) must mathematically decrease.

Tasks:
1. Extract the underlying numerical data matrix from your `evoked_dirty` and `evoked_clean` objects using the `.get_data()` method.
2. Compute the global variance of both matrices using standard NumPy functions.
3. Written Response: State the two variance values. Mathematically, why must $Var(X_{clean})$ always be less than $Var(X_{dirty})$ if a component was successfully removed?
"""

In [ ]:
# 1. Extract the numerical data matrices (Channels x Time)
# TODO: Fill in the expected function
X_dirty = evoked_dirty.____()
X_clean = evoked_clean.____()

In [ ]:
# 2. Compute the global variance of the entire data matrix
#TODO: Fill in expected function
var_dirty = np.____(X_dirty)
var_clean = np.____(X_clean)

print(f"Global Variance (Dirty): {var_dirty:.5e}")
print(f"Global Variance (Clean): {var_clean:.5e}")

In [ ]:
#@title Exercise 4: Forward Modeling the Artifact
"""
We know the generative model for ICA is $X = AS$. By excluding a component, we set its specific row in the source matrix $S$ to zero before multiplying it back through the mixing matrix $A$.

What happens if we execute the inverse? What if we set *every* row in $S$ to zero *except* the eye blink component?

Think of this like reviewing game tape focused exclusively on the opposing blockers, blurring out the rest of the court. By isolating the artifact's spatial projection back to the scalp, we can see exactly what voltage the ICA algorithm is subtracting from our dirty data.

Tasks
1. Create a list containing all 15 component indices *except* your EOG component.
2. Assign this massive list to `ica.exclude`.
3. Apply this to a copy of your epochs to create `epochs_blink_only`.
4. Plot the average Evoked response of this isolated artifact.
5. Written Response: Compare this plot to your `evoked_dirty` plot. What does this demonstrate about the mathematical linearity of the ICA unmixing process?
"""

In [ ]:
# 1. Create a list of all components EXCEPT the EOG component
all_components = list(range(15))
# TODO: Complete loop. Use list comprehension to exclude the blink index
blink_only_exclude = [c for c in all_components if c != ____]

# 2. TODO: Assign the new exclusion list
ica.exclude = ____

In [ ]:
# 3. TODO: Apply the ICA spatial filter to a copy of the original epochs
epochs_blink_only = ica.____(epochs.copy())

# 4. Calculate the average and plot [Use Auditory/Right]
evoked_blink_only = epochs_blink_only['...'].average()
evoked_blink_only.plot(spatial_colors=True, titles='Evoked: Blink Forward Model (Auditory Right)')

In [ ]:
#@title Exercise 5: Frequency Verification in Sensor Space
"""
Visualizing topomaps and time-series plots is crucial, but we must also verify our cleaning in the frequency domain.

The blink artifact lives primarily in the Delta and Theta frequency bands (1-5 Hz). If our ICA spatial filter was successful, we should see a massive drop in low-frequency power at the frontal electrodes, but we should *not* see a drop in low-frequency power at the occipital (visual) electrodes.

Tasks
1. Compute the Power Spectral Density (PSD) for a frontal electrode (`EEG 001`) from both your dirty and clean Evoked objects.
2. Compute the PSD for an occipital electrode (`EEG 059`) from both your dirty and clean Evoked objects.
3. Plot the comparisons.
4. Written Response: Describe the difference in low-frequency power (1-5 Hz) between the dirty and clean data at the frontal versus occipital electrodes. Does this confirm that our spatial filter preserved the brain data?
"""

In [ ]:
# 1. Compute PSD for the frontal electrode (EEG 001) before and after ICA
psd_frontal_dirty = evoked_dirty.compute_psd(fmax=40, picks=['EEG 001'])

# TODO: Apply the same computation to the clean data
psd_frontal_clean = evoked_clean.____(fmax=40, picks=['EEG 001'])

In [ ]:
# 2. Compute PSD for the occipital electrode (EEG 059) before and after ICA
# TODO: Check the channel name
psd_occipital_dirty = evoked_dirty.compute_psd(fmax=40, picks=[____])
psd_occipital_clean = evoked_clean.compute_psd(fmax=40, picks=['EEG 059'])

In [ ]:
# 3. Plotting the comparisons (Execution code provided)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frontal Plot
psd_frontal_dirty.plot(axes=axes[0], color='red', show=False, average=True)
psd_frontal_clean.plot(axes=axes[0], color='blue', show=False, average=True)
axes[0].set_title('Frontal (EEG 001): Dirty (Red) vs Clean (Blue)')

# Occipital Plot
psd_occipital_dirty.plot(axes=axes[1], color='red', show=False, average=True)
psd_occipital_clean.plot(axes=axes[1], color='blue', show=False, average=True)
axes[1].set_title('Occipital (EEG 059): Dirty (Red) vs Clean (Blue)')

plt.tight_layout()
plt.show()